# Classificador de Máxima Correlação

*Implementação em numpy · base processada (sem duplicatas, z-score)*

Cada classe é representada pelo seu centroide $\mathbf{m}_i$. A amostra é atribuída à classe de maior correlação (cosseno do ângulo) com ela:

$$\hat{y}(\mathbf{x}) = \arg\max_i \frac{\mathbf{x}^\top \mathbf{m}_i}{\|\mathbf{x}\|\,\|\mathbf{m}_i\|}$$

Como $\|\mathbf{x}\|$ é o mesmo para todas as classes, basta normalizar os centroides.

Protocolo (OBS 1–4 do enunciado): 100 rodadas treino-teste independentes e estratificadas para cada separação 20/80, 30/70, 50/50, 70/30 e 80/20.

## Dados

In [ ]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"  # antes do numpy: evita falta de memória do OpenBLAS no Windows

import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
from avaliacao import (SPLITS, carregar, avaliar, estatisticas, matrizes_confusao, grafico_splits,
                       por_classe, grafico_por_classe, resumo, salvar)

pd.set_option("display.max_rows", 200)

X, y, classes = carregar(processada=True)
print(f"{len(y)} instâncias | {X.shape[1]} atributos | {len(classes)} classes: {list(classes)}")

## Modelo

In [ ]:
class MaximaCorrelacao:
    def fit(self, X, y):
        self.classes_ = np.unique(y)
        C = np.array([X[y == c].mean(axis=0) for c in self.classes_])
        self.prototipos_ = C / np.linalg.norm(C, axis=1, keepdims=True)  # centroides com norma unitária
        return self

    def predict(self, X):
        return self.classes_[(X @ self.prototipos_.T).argmax(axis=1)]  # maior cosseno


MODELOS = {"Máxima correlação": MaximaCorrelacao()}

## Execução
100 rodadas × 5 separações. Em cada rodada, todos os modelos usam a mesma divisão treino/teste.

In [ ]:
res, confusoes = avaliar(MODELOS, X, y, classes)
salvar(res, "classification_processed", "max_correlacao")

## OBS 1 e 2 — Estatísticas de acurácia e tempos de treinamento/teste (ms)

In [ ]:
estatisticas(res).round(4)

## OBS 2 — Matrizes de confusão da melhor e da pior rodada

In [ ]:
matrizes_confusao(res, confusoes, classes, splits=SPLITS)

## OBS 3 — Efeito da separação treino/teste

In [ ]:
grafico_splits(res)

## OBS 4 — Desempenho por classe (taxa de acerto de cada classe)

In [ ]:
por_classe(res, classes).round(4)

In [ ]:
grafico_por_classe(res, classes)

## Resumo dos resultados

In [ ]:
resumo(res, classes)